# Graph Summaries over a Binary

This notebook demonstrates how to create hierachical summaries over a function call graph extracted from a binary.

## Load Data

### Load and Visualize the Graph with ipysigma


In [1]:
#from idadex import idaapi
import sys
from typing import Hashable
from networkx import k_core, DiGraph
import importlib
importlib.invalidate_caches()
from ipysigma import Sigma
import pandas as pd
import networkx as nx
import src.visualize_cfg as cfg
import os
import src.hierarchical_summaries as st
from src.graph_viz_layout import function_cluster_layout, add_force_weights
from src.function_call_graph import get_level0_summary_graph
from src.hierarchical_summaries import HierarchicalGraphSummarizer
import igraph as ig
import leidenalg as la
import src.summarize_graph as sg
importlib.reload(sg)


json_path = "reorder_and_pad2.json"

def prune_graph(og: DiGraph[Hashable]) -> DiGraph[Hashable]:
    to_return = og.copy()
    print(f"Graph loaded: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")

    # remove self loops
    to_return.remove_edges_from(nx.selfloop_edges(to_return))

    to_return = collapse_thunks(to_return)
    to_return = cfg.collapse_chains(to_return)

    # remove nodes with degree <= 1 - ALSO REMOVING ISOLATED LEAVES
    #to_return = k_core(to_return, 2)
    #entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    #print(f"entrypoint is {entrypoint}")
    # to_be_removed = [x for  x in G.nodes() if G.degree()[x] <= 1]
    # print(f"Number of nodes to be removed: {len(to_be_removed)}")
    # G.remove_nodes_from(to_be_removed)
    # # Basic info
    # print(f"Number of functions after removing degree <= 1: {len(G.nodes)}")
    #
    # print(f"Number of nodes after collapsing chains: {len(G.nodes)}")
    # entrypoint = [x for x in G.nodes() if G.nodes[x]['entry_point'] == True]
    # if not entrypoint:
    #     print("No entrypoint found. Please check the graph.")
    #     raise ValueError("No entrypoint found")
    print(f"Graph pruned: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    return to_return

def collapse_thunks(G: nx.DiGraph):
    collapsed_G = G.copy()
    nodes_to_process = list(collapsed_G.nodes())

    for node in nodes_to_process:

        flags = collapsed_G.nodes[node].get('flags', {})
        if flags & 128:
            preds = list(collapsed_G.predecessors(node))
            succs = list(collapsed_G.successors(node))
            if len(succs) == 1:
                collapsed_G.remove_node(node)
                for pred in preds:
                    collapsed_G.add_edge(pred, succs[0])
    return collapsed_G

# Load the graph data
G = cfg.load_cfg(cfg.read_json(json_path))
if G is None:
    print(f"Could not load graph from {json_path}. Please make sure the file exists and it is a valid CFG JSON.")
    exit()

assert(G is not None)
pruned = prune_graph(G)

largest_degree = 0
def set_size(n, d) -> int:
    return len(d.get('instrs'))+G.out_degree(n)

Graph loaded: 5174 nodes, 8729 edges
entrypoint is ['0x1400155d8']
Graph pruned: 5025 nodes, 8473 edges


Ways to reduce:
- strongly connected components
- remove nodes with degree <= 1
- clique detection and collapse
- graph community collapse
min cuts
- graph spectral?
- laplacian?
- adamic-adar index?

In [ ]:
## needed to reload code changed on disk, this is the correct import
summaries = sg.summarize_graph(pruned, base_url="http://192.168.1.101:8000/v1", max_concurrent=256, model="qwen3-coder-next", cache_path="./summarized_with_edge_types.json")


Summarizing functions:  79%|███████▉  | 265/335 [00:00<?, ?func/s]

### Load Summaries

In [2]:
import jsonlines
summaries = dict()
with jsonlines.open("cache_full_func.json", "r") as f:
    for line in f:

        summaries.update(line)

In [3]:
for node in pruned.nodes():
    pruned.nodes[node]['summary'] = summaries.get(pruned.nodes[node].get('func'))

In [ ]:
# Print all possible edges type and count
from collections import Counter

Counter(d.get("type", "unknown") for _, _, d in pruned.edges(data=True))

Counter({'JUMP_NEAR': 3824,
         'ORDINARY_FLOW': 3708,
         'call': 799,
         'unknown': 73,
         'OFFSET': 66,
         'exception': 3})

## Control-Flow-Graph (Block Level)

In [5]:
# visualize with ipysigma, now with summaries
weighted_g = add_force_weights(pruned)

layout = function_cluster_layout(weighted_g,radius=12, spacing=120)

widget = Sigma(
    weighted_g,
    layout=layout,
    start_layout=False,
    node_label="label",
    raw_node_color=lambda n, d: d.get("color"),
    edge_color="type",
    default_edge_type="arrow",
    height=800
)
display(widget)

Sigma(nx.DiGraph with 5,025 nodes and 8,473 edges)

## Function Call-graph 

In [6]:
level0_summary_graph = get_level0_summary_graph(pruned)

In [7]:
# Visualise function call graph
widget = Sigma(
    level0_summary_graph,
    start_layout=False,
    node_label="name",
    raw_node_color=lambda n, d: d.get("color"),
    edge_color="type",
    default_edge_type="arrow",
    height=800
)
display(widget)


Sigma(nx.DiGraph with 335 nodes and 522 edges)

**Observations**

* **4 important sink functions (out_degree == 0 & in_degree high):** This includes utility functions like loggers, memory handlers (`memcpy`), or visualization routines. This high in-degree significantly impacts how we summarize function interactions, especially using traditional clustering algorithms like Louvain and Leiden. Because those algorithms are density-based and generally treat the graph as undirected, these sinks have a high chance of becoming artificial "center hubs" and grouping functions together that don't actually interact in the real execution flow.

* **Main graph structures:** Looking at the overall topology, the main structures we see throughout the graph are **Trees** (linear execution paths) and **Cycles** (mutually recursive function loops).

## Hierarchical Summaries

### Try original community detection (Leiden & Louvain) over call graph

#### Louvain

In [8]:
# Remove isolated nodes
cleaned_function_graph = HierarchicalGraphSummarizer(level0_summary_graph)
cleaned_function_graph.remove_isolated_nodes()
graph = cleaned_function_graph.graph

[initial] Removed 49 isolated node(s)


In [9]:
communities_louvain = nx.community.louvain_communities(graph, weight='call', seed=42) #weight is the name of the edge that should be look into

In [10]:
# Extract the node with the highest degree in the subgraph to see what could potentially centre hub
for i, comm in enumerate(communities_louvain):
    # Create a subgraph of just this community
    subgraph = graph.subgraph(comm)
    
    # Find the node with the highest degree within this community
    nodes_by_degree = sorted(subgraph.degree(), key=lambda x: x[1], reverse=True)
    leader_node = nodes_by_degree[0][0]
    
    print(f"Community {i} was structurally driven by Node {leader_node} (Internal Degree: {nodes_by_degree[0][1]})")
    print(f"Number of functions in this community: {len(comm)}")
    print(f"Summary of the highest degree node: {subgraph[leader_node]}")

Community 0 was structurally driven by Node memcpy (Internal Degree: 22)
Number of functions in this community: 46
Summary of the highest degree node: {}
Community 1 was structurally driven by Node sub_140001A90 (Internal Degree: 8)
Number of functions in this community: 31
Summary of the highest degree node: {'sub_1400054E4': {'type': 'call', 'edge_types': ['call'], 'call_count': 1, 'call_sites': [{'src_block': '0x140001a90', 'dst_block': '0x1400054fc', 'src_func': 'sub_140001A90', 'dst_func': 'sub_1400054E4', 'type': 'call', 'conditional': False}], 'conditional': False}, 'sub_1400018C0': {'type': 'call', 'edge_types': ['call'], 'call_count': 1, 'call_sites': [{'src_block': '0x140001b10', 'dst_block': '0x1400018c0', 'src_func': 'sub_140001A90', 'dst_func': 'sub_1400018C0', 'type': 'call', 'conditional': False}], 'conditional': False}, 'sub_1400054C4': {'type': 'call', 'edge_types': ['call'], 'call_count': 1, 'call_sites': [{'src_block': '0x140001b94', 'dst_block': '0x1400054c4', 'src_

In [11]:
# Add summaries to the graph
for community_id, node_set in enumerate(communities_louvain):
    for node in node_set:
        if node in graph.nodes:
            # Add community ID to node
            graph.nodes[node]["community_louvain"] = community_id

In [12]:
len(communities_louvain)

12

In [13]:
# Technically, now that we remove the isolated nodes, should be none
count = 0 
for comm in communities_louvain:
    if len(comm) == 1: 
        count +=1

count

0

In [14]:
widget = Sigma(
    graph,
    start_layout=True,                
    node_label="name",
    node_color="community_louvain",           
    node_color_palette="Tableau10",  
    node_size=level0_summary_graph.degree, 
    edge_color="type",
    default_edge_type="arrow",
    height=800
)
display(widget)

Sigma(nx.DiGraph with 286 nodes and 522 edges)

#### Leiden

In [15]:

def get_leiden_communities(nx_graph: nx.DiGraph) -> list[set]:
    # Translate NetworkX structure to igraph
    ig_graph = ig.Graph.from_networkx(nx_graph)
    
    # Execute Leiden optimization on the graph topology
    partition = la.find_partition(ig_graph, la.ModularityVertexPartition, seed=42)
    
    # Map internal integer IDs back to original function names
    communities = []
    for cluster in partition:
        mapped_nodes = set(ig_graph.vs[node_idx]["_nx_name"] for node_idx in cluster)
        communities.append(mapped_nodes)
        
    return communities

communities = get_leiden_communities(graph)


In [16]:
len(communities)

13

In [17]:
count = 0 
for comm in communities:
    if len(comm) == 1: 
        count +=1

count

0

In [18]:
for community_id, node_set in enumerate(communities):
    for node in node_set:
        if node in graph.nodes:
            # Add community ID to node
            graph.nodes[node]["community"] = community_id


In [19]:
widget = Sigma(
    graph,
    start_layout=True,                
    node_label="name",
    node_color="community",           
    node_color_palette="Tableau10",  
    node_size=level0_summary_graph.degree, 
    edge_color="type",
    default_edge_type="arrow",
    height=800
)
display(widget)

Sigma(nx.DiGraph with 286 nodes and 522 edges)

#### Compare Louvain and Leiden sets

In [20]:
print(communities == communities_louvain)

False


In [22]:
set_louvain = {frozenset(s) for s in communities_louvain}
set_leiden = {frozenset(s) for s in communities}

identical_sets = set_louvain.intersection(set_leiden)
unique_sets_louvain = set_louvain - set_leiden
unique_sets_leiden = set_leiden - set_louvain

print("Identical sets:", identical_sets)
print("Unique Louvain sets:", unique_sets_louvain)
print("Unique Leiden sets:",unique_sets_leiden)

Identical sets: {frozenset({'sub_14001240C', 'sub_140011760', 'sub_140010F68', 'sub_14001266C', 'sub_14001104C', 'sub_140014EDC', 'sub_1400135B8', 'sub_1400126E8', 'sub_140010EF8', 'sub_1400110C0', 'sub_140010FD8'}), frozenset({'sub_14001171C', 'sub_140012FFC', 'sub_140012EC8', 'sub_14000AF24'}), frozenset({'MmQuitNextSession', 'sub_140015A48', '___scrt_initialize_default_local_stdio_options', 'sub_140015A58'}), frozenset({'__ZN68_$LT$core..num..error..ParseIntError$u20$as$u20$core..fmt..Debug$GT$3fmt17h9bb60b2936353f33E', 'sub_140012D1C', 'sub_140011DE0', '_ZN64_$LT$core..str..error..Utf8Error$u20$as$u20$core..fmt..Debug$GT$3fmt17h65be6feb6292d73fE', 'sub_140011EA8', 'sub_1400130C8'}), frozenset({'sub_140011C78', '_ZN70_$LT$core..num..error..TryFromIntError$u20$as$u20$core..fmt..Debug$GT$3fmt17h3926fd60bb423624E', 'sub_140004AC8'})}
Unique Louvain sets: {frozenset({'sub_140016C2C', 'memcmp', 'sub_140008FF0', 'sub_14000F78C', 'sub_140011494', 'sub_14000FA0C', 'sub_140011134', 'sub_1400

In [24]:
# Look at the summaries for the identical sets
louvain_set_to_id = {frozenset(comm): idx for idx, comm in enumerate(communities_louvain)}
leiden_set_to_id = {frozenset(comm): idx for idx, comm in enumerate(communities)}

def _function_summaries(func_set):
    return "\n\n".join(
        f"{func}: {graph.nodes[func].get('summary', 'No summary available.')}"
        for func in sorted(func_set)
    )

rows = []
for set_idx, func_set in enumerate(sorted(identical_sets, key=lambda s: (len(s), sorted(s)))):
    rows.append({
        "set": set_idx,
        "func": "\n".join(sorted(func_set)),
        "louvain sum": _function_summaries(communities_louvain[louvain_set_to_id[func_set]]),
        "leiden sum": _function_summaries(communities[leiden_set_to_id[func_set]]),
    })

identical_summary_table = pd.DataFrame(rows, columns=["set", "func", "louvain sum", "leiden sum"])
display(identical_summary_table)


,set,func,louvain sum,leiden sum
0,0,_ZN70_$LT$core..num..error..TryFromIntError$u2...,_ZN70_$LT$core..num..error..TryFromIntError$u2...,_ZN70_$LT$core..num..error..TryFromIntError$u2...
1,1,MmQuitNextSession\n___scrt_initialize_default_...,MmQuitNextSession: None\n\n___scrt_initialize_...,MmQuitNextSession: None\n\n___scrt_initialize_...
2,2,sub_14000AF24\nsub_14001171C\nsub_140012EC8\ns...,sub_14000AF24: `sub_14000AF24` is a **structur...,sub_14000AF24: `sub_14000AF24` is a **structur...
3,3,_ZN64_$LT$core..str..error..Utf8Error$u20$as$u...,_ZN64_$LT$core..str..error..Utf8Error$u20$as$u...,_ZN64_$LT$core..str..error..Utf8Error$u20$as$u...
4,4,sub_140010EF8\nsub_140010F68\nsub_140010FD8\ns...,sub_140010EF8: This function (`sub_140010EF8`)...,sub_140010EF8: This function (`sub_140010EF8`)...


### Try to run hierarchical summaries over Leiden Communities

In [25]:
import json
from src.llm_interface import LLMInterface

community_llm = LLMInterface(
    base_url="http://192.168.1.101:8000/v1",
    model="qwen3-coder-next",
    max_concurrent=16,
)

comm_summaries = {}

PROMPT = """
You are a reverse engineer summarizing a community of functions that will be collapsed into one graph node.
Use only the supplied function summaries and edge lists. Do not hallucinate missing links or behavior.
Explain what this community appears to do, how its functions cooperate, and which external communities/functions it depends on or is called by.
Preserve important edge direction, edge type, conditionality, and call-site details when they affect the interpretation.
Write a concise summary useful to another reverse engineer reading the bundled community node.
""".strip()

async def get_comm_summaries(communities: list, graph, llm=community_llm):
    node_to_community = {
        node: community_id
        for community_id, comm in enumerate(communities)
        for node in comm
    }

    community_summaries = {}
    for community_id, comm in enumerate(communities):
        subgraph = graph.subgraph(comm).copy()
        external_links = identify_external_edges(
            graph,
            comm,
            node_to_community=node_to_community,
            community_id=community_id,
        )

        summary = await llm.call_llm(PROMPT, comm_prompt(subgraph, external_links))

        community_summaries[community_id] = {
            "community_id": community_id,
            "nodes": list(comm),
            "external_links": external_links,
            "summary": summary,
        }

    return community_summaries

def comm_prompt(subgraph, external_links=None):
    functions = []
    for node, data in subgraph.nodes(data=True):
        functions.append({
            "node": node,
            "name": data.get("name", data.get("func", str(node))),
            "summary": data.get("summary") or "No summary available.",
            "callers": data.get("callers", []),
            "callees": data.get("callees", []),
        })

    internal_edges = []
    for src, dst, data in subgraph.edges(data=True):
        internal_edges.append({
            "source": src,
            "target": dst,
            "type": data.get("type", "unknown"),
            "edge_types": data.get("edge_types", [data.get("type", "unknown")]),
            "call_count": data.get("call_count", 1),
            "conditional": data.get("conditional", False),
            "call_sites": data.get("call_sites", []),
        })

    payload = {
        "functions": functions,
        "internal_edges": internal_edges,
        "external_edges": external_links or [],
    }

    return (
        "Summarize this function community from the structured graph data below. "
        "Internal edges connect functions inside this community. External edges connect this community "
        "to functions in other communities and must be kept as explicit dependencies or callers.\n\n"
        + json.dumps(payload, indent=2, default=str)
    )

def identify_external_edges(graph, community_nodes, node_to_community=None, community_id=None):
    community_nodes = set(community_nodes)
    external_links = []

    edge_iter = list(graph.edges(community_nodes, data=True))
    if graph.is_directed():
        edge_iter.extend(graph.in_edges(community_nodes, data=True))

    seen_edges = set()
    for src, dst, edge_data in edge_iter:
        edge_key = (src, dst)
        if edge_key in seen_edges:
            continue
        seen_edges.add(edge_key)

        if src in community_nodes and dst in community_nodes:
            continue

        external_links.append({
            "source": src,
            "target": dst,
            "source_community": community_id if src in community_nodes else node_to_community.get(src),
            "target_community": community_id if dst in community_nodes else node_to_community.get(dst),
            "edge_data": dict(edge_data),
        })

    return external_links

In [26]:
# Create new summaries
comm_summaries = await get_comm_summaries(communities=communities, graph=graph)

CancelledError: 

In [27]:
comm_summaries

{}

In [64]:
def generate_json_comm_file(comm_sum, path: str = "summaries_leiden_comm.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(comm_sum, f, ensure_ascii=False, indent=4)
    print(f"Summaries written to {path}")

In [65]:
generate_json_comm_file(comm_summaries)

Summaries written to summaries_leiden_comm.json


#### Visualize Leiden Community Summary Graph

In [28]:
import html
import json
import re
from collections import defaultdict
from IPython.display import HTML, display

def _summary_title(summary: str) -> str:
    for line in summary.splitlines():
        line = line.strip()
        if not line or line in {"---", "```"}:
            continue
        line = re.sub(r"^[#*\s]+|[*\s]+$", "", line)
        return line[:120]
    return "No summary available"

def _summary_excerpt(summary: str, limit: int = 420) -> str:
    text = re.sub(r"\s+", " ", summary).strip()
    return text[:limit] + ("..." if len(text) > limit else "")

def build_community_summary_graph(summary_path: str = "summaries_leiden_comm.json"):
    with open(summary_path, "r", encoding="utf-8") as f:
        community_summaries = json.load(f)

    H = nx.DiGraph()

    for raw_id, data in community_summaries.items():
        community_id = str(data.get("community_id", raw_id))
        nodes = data.get("nodes", [])
        summary = data.get("summary", "")
        title = _summary_title(summary)
        H.add_node(
            community_id,
            label=f"Community {community_id}",
            name=f"Community {community_id}",
            summary_title=title,
            summary=summary,
            legend=f"Community {community_id}: {title}",
            community_size=len(nodes),
            functions=", ".join(nodes[:20]) + (", ..." if len(nodes) > 20 else ""),
        )

    edge_acc = defaultdict(lambda: {
        "call_count": 0,
        "edge_types": set(),
        "links": [],
    })

    for data in community_summaries.values():
        for link in data.get("external_links", []):
            src_comm = link.get("source_community")
            dst_comm = link.get("target_community")
            if src_comm is None or dst_comm is None or src_comm == dst_comm:
                continue

            src_comm = str(src_comm)
            dst_comm = str(dst_comm)
            edge_data = link.get("edge_data", {})
            acc = edge_acc[(src_comm, dst_comm)]
            acc["call_count"] += edge_data.get("call_count", 1)
            acc["edge_types"].update(edge_data.get("edge_types", [edge_data.get("type", "unknown")]))
            acc["links"].append({
                "source": link.get("source"),
                "target": link.get("target"),
                "type": edge_data.get("type", "unknown"),
                "call_count": edge_data.get("call_count", 1),
            })

    for (src_comm, dst_comm), data in edge_acc.items():
        edge_types = sorted(t for t in data["edge_types"] if t)
        H.add_edge(
            src_comm,
            dst_comm,
            type=", ".join(edge_types) or "unknown",
            call_count=data["call_count"],
            label=f"{data['call_count']} call(s)",
            links=data["links"],
        )

    return H, community_summaries

def display_community_legend(community_summaries):
    rows = []
    for raw_id, data in sorted(community_summaries.items(), key=lambda item: int(item[0])):
        community_id = data.get("community_id", raw_id)
        nodes = data.get("nodes", [])
        summary = data.get("summary", "")
        rows.append(
            "<details style='margin: 8px 0; padding: 8px; border: 1px solid #ddd; border-radius: 6px;'>"
            f"<summary><b>Community {html.escape(str(community_id))}</b> "
            f"<span style='color:#666'>({len(nodes)} functions)</span> - "
            f"{html.escape(_summary_title(summary))}</summary>"
            f"<p style='margin: 8px 0 0; line-height: 1.4'>{html.escape(_summary_excerpt(summary))}</p>"
            "</details>"
        )
    display(HTML("<h3>Community Legend</h3>" + "".join(rows)))


In [30]:
community_graph, community_summary_data = build_community_summary_graph("summaries_leiden_comm.json")

widget = Sigma(
    community_graph,
    start_layout=True,
    node_label="label",
    node_color="community_size",
    node_color_palette="Tableau10",
    node_size="community_size",
    edge_color="type",
    edge_size="call_count",
    default_edge_type="arrow",
    height=700,
)
display(widget)
display_community_legend(community_summary_data)

Sigma(nx.DiGraph with 13 nodes and 30 edges)

### Test with custom pipeline

In [ ]:
summarizer = HierarchicalGraphSummarizer(level0_summary_graph)

In [ ]:
await summarizer.execute_pipeline()